# Data Loading


In [1]:
import pandas as pd

DATA_PATH = "issue_repo_pre_estimation.csv"

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(53169, 31)


,issue_id,jira_id,issue_key,issue_url,project_id,project_key,project_name,repository_id,repository_name,repository_url,...,assignee_id,title_changed_after_estimation,description_changed_after_estimation,story_point_changed_after_estimation,has_description_code,num_components,component_names,num_affected_versions,affected_version_names,story_point
0,65,77638,XD-3768,https://jira.spring.io/rest/api/2/issue/77638,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,0,NaN,0,NaN,1.0
1,66,77511,XD-3767,https://jira.spring.io/rest/api/2/issue/77511,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,1,0,NaN,2,1.3 GA | 1.3.2,1.0
2,67,77130,XD-3766,https://jira.spring.io/rest/api/2/issue/77130,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,1,Stream Module,1,1.3.1,10.0
3,68,71950,XD-3765,https://jira.spring.io/rest/api/2/issue/71950,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,71.0,1,0,0,0,0,NaN,1,1.3.1,8.0
4,69,71805,XD-3764,https://jira.spring.io/rest/api/2/issue/71805,1,XD,Spring XD,1,Spring,https://jira.spring.io/,...,NaN,0,0,0,0,1,Batch,0,NaN,5.0


In [2]:
# Keep only rows with valid labels and valid input text
df = df[df["story_point"].notna()]
df = df[df["input_text"].notna()]
df = df[df["input_text"].astype(str).str.strip().str.len() > 0]

# Make sure label is float for regression
df["story_point"] = df["story_point"].astype(float)

print(df.shape)

(53169, 31)


In [3]:
# Categorical/text feature columns
text_cols = [
    "project_key",
    "project_name",
    "repository_name",
    "issue_type",
    "priority",
    "component_names",
    "affected_version_names"
]

for col in text_cols:
    df[col] = df[col].fillna("Unknown").astype(str)

# More meaningful names for missing list fields
df["component_names"] = df["component_names"].replace("Unknown", "None")
df["affected_version_names"] = df["affected_version_names"].replace("Unknown", "None")

# Numeric/count columns
numeric_cols = [
    "num_components",
    "num_affected_versions",
    "has_description_code"
]

for col in numeric_cols:
    df[col] = df[col].fillna(0)

# ID columns
id_cols = [
    "sprint_id",
    "creator_id",
    "reporter_id",
    "assignee_id"
]

for col in id_cols:
    df[col] = df[col].fillna(-1)

In [4]:
df["text_only_input"] = df["input_text"].astype(str)

In [5]:
print(df["text_only_input"].iloc[37920][:1000])

"CLONE - Remove old class renames" """core_component looks for COMPONENT/db/renamedclasses.php files, it allows us to avoid traumatic transitions and BC breakages by listing classes that have been removed.    We need to clean the list at some point."""


In [6]:
df["creation_date"] = pd.to_datetime(df["creation_date"], errors="coerce")

# Drop rows where creation date is missing, if any
df = df[df["creation_date"].notna()]

# Sort chronologically
df = df.sort_values("creation_date").reset_index(drop=True)

n = len(df)

train_end = int(0.70 * n)
val_end = int(0.80 * n)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print(train_df["creation_date"].min(), "to", train_df["creation_date"].max())
print(val_df["creation_date"].min(), "to", val_df["creation_date"].max())
print(test_df["creation_date"].min(), "to", test_df["creation_date"].max())

Train: (37218, 32)
Val: (5317, 32)
Test: (10634, 32)
2004-12-21 16:28:52 to 2018-04-24 00:36:48
2018-04-24 00:37:12 to 2019-01-29 21:17:08
2019-01-29 21:49:05 to 2020-10-22 02:06:10


In [7]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [8]:
TEXT_COLUMN = "text_only_input"

#Model

In [9]:
train_data = train_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

val_data = val_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

test_data = test_df[[TEXT_COLUMN, "story_point"]].rename(
    columns={TEXT_COLUMN: "text", "story_point": "label"}
)

In [10]:
train_data["label"] = train_data["label"].astype("float32")
val_data["label"] = val_data["label"].astype("float32")
test_data["label"] = test_data["label"].astype("float32")

In [14]:
import requests
import re
import time

def get_ollama_prediction(text, model="llama3"):
    prompt = f"""
You are a software engineering expert.

Estimate the story points for the following issue.
Return ONLY a single number (no explanation).

{text[:1000]}
"""

    try:
        response = requests.post(
            "http://localhost:11434/api/generate",
            json={
                "model": model,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0
                }
            },
            timeout=360
        )

        output = response.json()["response"].strip()

        match = re.search(r"\d+(\.\d+)?", output)
        return float(match.group()) if match else None

    except Exception as e:
        print("Error:", e)
        return None

In [12]:
subset = test_data.sample(100, random_state=42).copy()

In [15]:
print(get_ollama_prediction(subset["text"].iloc[0]))

3.0


In [22]:
print(test_data["label"].iloc[0])

1.0


In [16]:
preds = []

for i, text in enumerate(subset["text"]):
    pred = get_ollama_prediction(text)
    preds.append(pred)

    if i % 10 == 0:
        print(f"Processed {i} samples")

    time.sleep(0.5)

Processed 0 samples
Processed 10 samples
Processed 20 samples
Processed 30 samples
Processed 40 samples
Processed 50 samples
Processed 60 samples
Processed 70 samples
Processed 80 samples
Processed 90 samples


# Analysis

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

y_true = subset["label"]

mae = mean_absolute_error(y_true, preds)
rmse = np.sqrt(mean_squared_error(y_true, preds))

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE:  6.0870
RMSE: 7.0819


In [30]:
adjusted_preds = []

for a in preds:
    if a == 3.0:
        adjusted_preds.append(1.0)
    elif a == 8.0:
        adjusted_preds.append(3.0)
    elif a == 13.0:
        adjusted_preds.append(8.0)
    else: 
        adjusted_preds.append(13.0)

adjusted_preds = preds - np.mean(preds) + np.median(train_data["label"])


mae = mean_absolute_error(y_true, adjusted_preds)
rmse = np.sqrt(mean_squared_error(y_true, adjusted_preds))

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")

MAE:  3.0302
RMSE: 5.0082
